# Проверка выкладок Кузьменко, волна 2 (XV–XXII) + вывод β для дефекта
1 ноября 2026. Ядро SageMath 9.6. Каждый блок: формулы → численная проверка (OK/FAIL, невязка).

In [ ]:
import numpy as np, math
trapz = getattr(np,'trapezoid',None) or np.trapz
RES={}
def ok(name,res,tol):
    res=float(res); s='OK' if abs(res)<tol else 'FAIL'; RES[name]=s
    print(f'{s:4} {name}: residual={res:.3e}')

## 1. Отклик κ (XV §5.4, XXI ур.4)
Баланс для сферы в ускоренном потенциальном потоке:
$\beta\dot v = a_m + c_a(a_m-\dot v)\;\Rightarrow\;\dot v=\kappa a_m,\quad \kappa=\dfrac{1+c_a}{\beta+c_a}$.
Пустая полость $\beta=0$, $c_a=\tfrac12$ → $\kappa=3$; единичный отклик требует $\beta=1$.

In [ ]:
for ca in (0.5,0.2,1.0):
    for beta in (0,0.25,1,2):
        am=1.7; vd=(1+ca)*am/(beta+ca)
        ok(f'balance ca={ca} beta={beta}', beta*vd-(am+ca*(am-vd)), 1e-12)
ok('sphere beta=0 -> kappa=3', 1.5/0.5-3, 1e-15)
ok('beta=1 -> kappa=1', 1.5/1.5-1, 1e-15)

## 2. Присоединённая масса сферы $c_a=1/2$
$\phi=-Ua^3\cos\theta/(2r^2)$, $K=\tfrac12\rho\int|\nabla\phi|^2dV=\tfrac12(c_a\rho V)U^2$.

In [ ]:
r=np.geomspace(1,2000,200001); th=np.linspace(0,np.pi,2001)
rad=trapz((1/(2*r**3))**2*r**2,r)
ang=trapz((4*np.cos(th)**2+np.sin(th)**2)*np.sin(th),th)*2*np.pi
ok('c_a = 1/2', 0.5*rad*ang/(0.5*4*np.pi/3)-0.5, 1e-6)

## 3. Профиль «реки» $u=-\sqrt{2E/r}$ (XXI §5.1)
$D_tu=u\,u'=-E/r^2$ — точно. Но $\nabla\!\cdot u=-3\lambda_m$, $\lambda_m=\sqrt{E/(2r^3)}$ ≠ 0 → нужен сток $\sigma=3\lambda_m$
(автор: «рождение новых атомов», закона нет). Оценка для Земли ниже.

In [ ]:
E=3.986004418e14; Rb=6371e3; h=1.0
u=lambda r:-np.sqrt(2*E/r)
Dtu=u(Rb)*(u(Rb+h)-u(Rb-h))/(2*h)
ok('D_t u = -E/r^2 (rel)', (Dtu+E/Rb**2)/(E/Rb**2), 1e-8)
div=((Rb+h)**2*u(Rb+h)-(Rb-h)**2*u(Rb-h))/(2*h)/Rb**2; lm=np.sqrt(E/(2*Rb**3))
ok('div u = -3 lambda_m (rel)', (div+3*lm)/(3*lm), 1e-8)
print('sigma =',3*lm,'1/s ; 1/sigma =',1/(3*lm),'s ; Q(R_E) =',4*np.pi*Rb**2*abs(u(Rb)),'m^3/s')

## 4. Кумулятивный фронт (XV ур.3,7–9)
$v^2R^3=C=E\ell^2\Rightarrow \ddot R=-3C/(2R^4)$; равенство с $-E/R^2$ при $R_*=\ell\sqrt{3/2}$; при $\ell=R_b$ у поверхности отношение 1.5.

In [ ]:
ok('R* = l sqrt(3/2)', 3/(2*1.5)-1, 1e-15)
ok('a_front/a_body at l=Rb = 1.5', (3*E*Rb**2/(2*Rb**4))/(E/Rb**2)-1.5, 1e-12)
ok('Earth a_front(Rb)=14.730', 1.5*E/Rb**2-14.730, 1e-3)

## 5. Гравиметрия по ветвям (XXI ур.23, XXII ур.13)
$\ddot z=-g+\varphi\dot z$; квадратичный МНК по каждой ветви при равных шагах по высоте даёт $|g_\uparrow-g_\downarrow|=\tfrac47|\lambda|gT$.
Проверка прямым моделированием траектории.

In [ ]:
g=9.8; T=0.3039; phi=1e-6; N=400001
t=np.linspace(0,T,N); dt=t[1]; z=np.zeros(N); v=np.zeros(N); v[0]=g*T/2
for i in range(N-1):
    vm=v[i]+(-g+phi*v[i])*dt/2; z[i+1]=z[i]+vm*dt; v[i+1]=v[i]+(-g+phi*vm)*dt
im=int(np.argmax(z))
def gfit(ts,zs):
    lev=np.linspace(zs.min(),zs.max(),3000)[1:-1]; o=np.argsort(zs)
    tt=np.interp(lev,zs[o],ts[o]); return -2*np.polyfit(tt,lev,2)[0]
c=abs(gfit(t[:im],z[:im])-gfit(t[im:],z[im:])); pred=4/7*phi*g*T
ok('branch contrast vs 4/7 lambda g T (rel)', (c-pred)/pred, 2e-2)
print('sim',c*1e8,'uGal ; formula',pred*1e8,'uGal')

## 6. Критерий пассивной совместимости (XVIII ур.4–5)
$\exists K=K^T\succeq0:\;Kw_0=d \iff w_0\!\cdot d>0$ (или $d=0$). Свидетель $K_0=dd^T/\sigma$.

In [ ]:
rng=np.random.default_rng(1); bad=0
for k in range(2000):
    w=rng.normal(size=3); d=rng.normal(size=3); s=w@d
    if s>1e-3:
        K=np.outer(d,d)/s; ev=np.linalg.eigvalsh(K)
        bad+= int(np.linalg.norm(K@w-d)>1e-9*np.linalg.norm(d) or ev.min()<-1e-12*ev.max())
ok('PSD witness', bad, 0.5)
w=np.array([1.,0,0]); d=np.array([-1.,0.3,0])
print('sigma<0 -> residual bound >=', max(0,-(w@d)/np.linalg.norm(w)))

## 7. Вывод β для дефекта нашей среды
**Идея.** В нашей модели частица — дефект упругой среды, «вытесненного объёма» нет; инерция целиком от собственного поля.
Аналог Кузьменко: $\beta_{\rm eff}\equiv m c^2/E_0$ (инерция / энергия покоя поля, в единицах $c^2$).
Тогда эквивалентность «вес = инерция» эквивалентна $\beta_{\rm eff}=1$ **при условии**, что гравитационная связь ∝ $E_0/c^2$ (это отдельный постулат).

**7a. Винтовая дислокация** (одна скорость $c_t$): поле при скорости $v$ — лоренц-сжатое статическое,
$E(v)=\gamma E_0$, $p=\gamma (E_0/c_t^2)v$ ⇒ $\beta_{\rm eff}=1$ **точно** (Франк 1949).

In [ ]:
mu=rho=c=b=1.0; r0=0.2; Rm=10.0; n=1201
s_=np.linspace(-Rm,Rm,n); ds=s_[1]-s_[0]; X,Y=np.meshgrid(s_,s_,indexing='ij'); rr=np.hypot(X,Y)
mask=(rr>r0)&(rr<Rm); rr[rr==0]=1
def screw(vv):
    gm=1/np.sqrt(1-vv**2)
    # u_z = (b/2pi) atan2(y, gm*x);  gradient in lab coords (x=gm*xi)
    xi=X/gm; den=(gm*xi)**2+Y**2
    ux=-(b/2/np.pi)*gm*Y/den; uy=(b/2/np.pi)*gm*xi/den
    m2=( (xi*gm)**2+Y**2 )   # region mask in lab-frame contracted coords
    msk=(np.hypot(gm*xi,Y)>r0)&(np.hypot(gm*xi,Y)<Rm)
    W=np.sum(((mu/2)*(ux**2+uy**2)+(rho/2)*vv**2*ux**2)[msk])*ds*ds/gm
    P=np.sum((rho*vv*ux**2)[msk])*ds*ds/gm
    return W,P
E0,_=screw(0.0)
for vv in (0.3,0.6,0.9):
    W,P=screw(vv); gm=1/np.sqrt(1-vv**2)
    ok(f'screw E=gamma E0, v={vv}', W/(gm*E0)-1, 5e-3)
    ok(f'screw p=gamma (E0/c^2) v, v={vv}', P/(gm*E0*vv)-1, 5e-3)

**7b. Краевая дислокация** (две скорости $c_t, c_l$), медленное скольжение (Хирт–Лоте):
$m=\dfrac{E_0}{c_t^2}(1-\nu)\Big[1+\big(\tfrac{c_t}{c_l}\big)^4\Big]$, $\big(\tfrac{c_t}{c_l}\big)^2=\dfrac{1-2\nu}{2(1-\nu)}$ ⇒ $\beta_{\rm eff}\ne1$.
Численно: кинетическая энергия поля $\tfrac12\rho\int(\partial_x u)^2dA$.

In [ ]:
def edge(nu):
    x=X; y=Y; r2=x**2+y**2
    ux_x=(b/2/np.pi)*(-y/r2 + y*(y**2-x**2)/(2*(1-nu)*r2**2))
    uy_x=-(b/2/np.pi)*((1-2*nu)/(4*(1-nu))*2*x/r2 + (2*x*r2-(x**2-y**2)*2*x)/(4*(1-nu)*r2**2))
    m=rho*np.sum((ux_x**2+uy_x**2)[mask])*ds*ds
    E0e=mu*b**2/(4*np.pi*(1-nu))*np.log(Rm/r0)
    return m/E0e
for nu in (0.0,0.25,1/3,0.45,0.497):
    k4=((1-2*nu)/(2*(1-nu)))**2; an=(1-nu)*(1+k4)
    ok(f'edge beta_eff, nu={nu:.3f} (an={an:.4f})', edge(nu)/an-1, 1e-2)

**7c. Glide-петля (смешанный характер), грубое сегментное усреднение** при $c_L\approx13c$ (наша оценка ⇒ $\nu\approx0.497$):
$\beta_{\rm eff}^{\rm loop}\approx\dfrac{\tfrac12+\tfrac12(1+k^4)}{\tfrac12+\tfrac12/(1-\nu)}$.

In [ ]:
cl=13.0; k2=1/cl**2; nu=(1-2*k2)/(2*(1-k2))   # from (ct/cl)^2=(1-2nu)/(2(1-nu))
num=0.5+0.5*(1+k2**2); den=0.5+0.5/(1-nu)
print('nu =',nu,' beta_eff(loop) ~',num/den)
print('Вывод: beta_eff=1 только для винтовой компоненты; краевая даёт ~0.50, петля ~0.67')

## Сводка

In [ ]:
print(sum(v=='OK' for v in RES.values()),'/',len(RES),'OK')
for k,v in RES.items():
    if v!='OK': print('FAIL:',k)

# 8. β_eff мёбиус-кольца (1 ноя 2026)
**Модель.** Тонкая петля радиуса $R$, касательная $e_t$. Вектор Бюргерса вращается по правилу $\psi=\chi/2+\varphi/2$ в одной из трёх плоскостей: (e_r,e_z), (e_r,e_t), (e_t,e_z).
Локально каждый элемент трактуется как прямая дислокация (приближение $\ln(R/r_0)\gg1$): винтовая часть $b_s=b\cdot e_t$, краевая часть $b_e$.
Энергия на единицу длины $\propto b_s^2+b_e^2/(1-\nu)$. Масса — тензор: винтовая $E_s/c^2$ изотропно поперёк линии; краевая — скольжение $m_g$ и переползание $m_c$.
$\beta_{ij}\equiv c^2 M_{ij}/E$. Скорость $c_L=13c\Rightarrow\nu=0.497$.

In [ ]:
cl=13.0; k2=1/cl**2; nu=(1-2*k2)/(2*(1-k2)); lam=1/k2-2
n=2001; Rm=10.0; r0=0.2
s=np.linspace(-Rm,Rm,n)+1e-7; ds=s[1]-s[0]; X,Y=np.meshgrid(s,s,indexing='ij'); R2=X**2+Y**2
msk=(np.sqrt(R2)>r0)&(np.sqrt(R2)<Rm); I=lambda F:np.sum(F[msk])*ds*ds
def smooth(x,y):
    rr=x**2+y**2
    return ((1/2/np.pi)*(x*y/(2*(1-nu)*rr)),
            -(1/2/np.pi)*((1-2*nu)/(4*(1-nu))*np.log(rr)+(x**2-y**2)/(4*(1-nu)*rr)))
h=1e-5
a=smooth(X+h,Y);bq=smooth(X-h,Y); dx=[(p-q)/(2*h) for p,q in zip(a,bq)]
a=smooth(X,Y+h);bq=smooth(X,Y-h); dy=[(p-q)/(2*h) for p,q in zip(a,bq)]
ux_x=dx[0]-(1/2/np.pi)*Y/R2; ux_y=dy[0]+(1/2/np.pi)*X/R2; uy_x=dx[1]; uy_y=dy[1]
exx,eyy,exy=ux_x,uy_y,0.5*(ux_y+uy_x)
Es=math.log(Rm/r0)/(4*np.pi)
Er=I(0.5*lam*(exx+eyy)**2+exx**2+eyy**2+2*exy**2)/Es
mg=I(ux_x**2+uy_x**2)/Es; mc=I(ux_y**2+uy_y**2)/Es
ok('E_edge/E_screw = 1/(1-nu)', Er*(1-nu)-1, 2e-3)
ok('m_glide c^2/E_screw = 1+(ct/cl)^4', mg/(1+k2**2)-1, 2e-3)
print('m_climb c^2/E_screw =',mc,' ; (m_g+m_c)/E_edge =',(mg+mc)/Er)

In [ ]:
N=4000; ph=(np.arange(N)+.5)*2*np.pi/N; dl=2*np.pi/N
er=np.stack([np.cos(ph),np.sin(ph),0*ph],1); et=np.stack([-np.sin(ph),np.cos(ph),0*ph],1); ez=np.tile([0,0,1.],(N,1))
def beta_of_b(bv):
    cs=np.sum(bv*et,1); be=bv-cs[:,None]*et; se=np.linalg.norm(be,axis=1)
    eh=be/np.maximum(se,1e-12)[:,None]; nc=np.cross(et,eh); E=np.sum(cs**2+se**2*Er)*dl; out=[]
    for V in np.eye(3):
        Vp=V-np.sum(V*et,1)[:,None]*et
        out.append(np.sum(cs**2*np.sum(Vp*Vp,1)+se**2*(mg*np.sum(Vp*eh,1)**2+mc*np.sum(Vp*nc,1)**2))*dl/E)
    return np.array(out)
def mobius(plane,chi=0.0):
    a=(chi+ph)/2; A,B={'rz':(er,ez),'rt':(er,et),'tz':(et,ez)}[plane]
    return np.cos(a)[:,None]*A+np.sin(a)[:,None]*B
for pl,name in (('rz','glide-край<->призм.край'),('rt','glide-край<->винт'),('tz','винт<->призм.край')):
    bx=beta_of_b(mobius(pl)); print(f'{pl} ({name}): beta_xx=beta_yy={bx[0]:.3f}, beta_zz={bx[2]:.3f}, avg={bx.mean():.3f}')
print('ref: обычная glide-петля b=const:',np.round(beta_of_b(np.tile([1,0,0.],(N,1))),3))
ok('orientation-avg beta = 2/3 (rz)', beta_of_b(mobius('rz')).mean()-2/3, 3e-3)
ok('orientation-avg beta = 2/3 (rt)', beta_of_b(mobius('rt')).mean()-2/3, 3e-3)
ok('orientation-avg beta = 2/3 (tz)', beta_of_b(mobius('tz')).mean()-2/3, 3e-3)

## 8b. Объяснение: теорема фон Лауэ (аналог «4/3»)
Для стационарной конфигурации $c^2M_{ij}=E\,\delta_{ij}+S_{ij}$, где $S_{ij}=\int T_{ij}\,dV$ — интегральное напряжение.
Петля без внешней опоры держится только натяжением линии, $T_{ij}\approx-\varepsilon\,t_it_j$, поэтому
$S_{ij}=-E\langle t_it_j\rangle=-\tfrac E2(\delta_{xx},\delta_{yy}),\;S_{zz}=0$ ⇒ $\beta=(\tfrac12,\tfrac12,1)$; среднее по ориентациям $1-\tfrac13=\tfrac23$.
Это и есть то, что получено выше (струна Намбу — точно; дислокации при $m_g+m_c=2E_e/c^2$ — тоже).
**β=1 изотропно ⇔ $S_{ij}=0$ ⇔ петля в механическом равновесии** (стресс Пуанкаре). Нужна стабилизирующая положительная добавка $+\tfrac E2$ в плоскости петли.

In [ ]:
S=-np.diag([.5,.5,0])
ok('Laue: Nambu loop beta=(1/2,1/2,1)', np.abs(np.diag(np.eye(3)+S)-beta_of_b(mobius('rz'))).max(), 3e-3)
# a stabilizing centrifugal stress from rotation/J: for a loop spinning in its plane, S_rot=+(1/2)M v^2 in-plane
for v2 in (0.0,0.5,1.0):
    print(f'v_rot^2/c^2={v2}: in-plane beta ~ {0.5+0.5*v2:.2f}  (beta=1 at v_rot=c - limit, i.e. needs a true Poincare stress)')

# 9. Стабилизация кольца и β=1 (1 ноя 2026)
**Лауэ + вириал масштабирования.** Для петли, единственная «пружина» которой — собственная энергия $E(R)$, в плоскости петли
$\operatorname{tr}_\parallel S=-R\,\dfrac{dE}{dR}$, $S_{zz}=0$ ⇒ $\langle\beta\rangle=1+\dfrac{\operatorname{tr}S}{3E}$.
Локально $E=2\pi R\varepsilon$ ⇒ $\operatorname{tr}S=-E$, $\langle\beta\rangle=2/3$ (§8).

**9a. Вращение кольца как твёрдого тела** невозможно: движение линии вдоль самой себя нефизично (калибровка Намбу) — центробежного напряжения нет.

**9b. Бегущий твист (фаза $\chi=\omega t$)** — «ток» на линии, как у вортона. Волна энергии $w$ на длину, групповая скорость $v$, импульс $p=w/v$.
Обручевый баланс: $\operatorname{tr}_\parallel S=-2\pi R(T-pv)/1\cdot\tfrac{1}{1}$ ⇒ равновесие при $pv=T$, т.е. $w=T\approx\varepsilon$ (киральный предел Картера).
Тогда $S=0$ ⇒ $\beta=1$ **изотропно и точно** (Лауэ для стационарной системы).

In [ ]:
def beta_twist(w, eps=1.0, T=1.0):
    E=eps+w; tr=-(T-w)          # per 2piR, p v = w
    b_in=1+tr/(2*E); b_z=1.0; return b_in,b_z,(2*b_in+b_z)/3
for w in (0,0.25,0.5,1.0):
    print(f'w/eps={w}:', [round(x,4) for x in beta_twist(w)])
ok('w=0 reproduces Laue loop (1/2,1)', beta_twist(0)[0]-0.5, 1e-15)
ok('chiral limit w=T gives beta=1', beta_twist(1)[2]-1, 1e-15)

**Связь с часами де Бройля.** Ранее доказано: часам нужно $E/(J\omega)=1$, что возможно только при скорости носителя $=c$.
В киральном пределе твист бежит со скоростью волны $v=c_t=c$ и несёт половину энергии.
Проверка: $J=pR\cdot2\pi R=wR\cdot2\pi R/c$, $\omega=c/R$ (одна волна на обход, мёбиус: $\psi=\varphi/2$ ⇒ $\omega_{\psi}=c/(2R)$).

In [ ]:
R=1.0; c=1.0; eps=1.0; w=1.0
E=2*np.pi*R*(eps+w); J=(w/c)*R*2*np.pi*R
for name,om in (('omega=c/R',c/R),('Mobius psi: omega=c/(2R)',c/(2*R))):
    print(f'{name}: E/(J omega) = {E/(J*om):.3f}')
print('=> E/(Jω)=1 не выполняется: статическая половина энергии не вращается (2 и 4).')

## 9c. Нелокальная поправка (полный $E(R)$)
Тонкое кольцо: $E(R)=\tfrac{\mu b^2R}{2}\big[\ln\tfrac{8R}{r_0}-C\big]$ (изотропная оценка; $C\approx2$, для вихревого кольца $7/4$).
$R\,dE/dR=E+\tfrac{\mu b^2R}{2}$ ⇒ $\langle\beta\rangle=\dfrac23-\dfrac{1}{3(\Lambda-C)}$, $\Lambda=\ln(8R/r_0)$.
Условие кирального равновесия ужесточается: $w=T_{\rm eff}=\varepsilon\,(1+\tfrac1{\Lambda-C})$.
Проверка вириала масштабирования — численным дифференцированием и прямой квадратурой энергии взаимодействия сегментов (Нейман, $\nu=0$).

In [ ]:
def E_neumann(R, r0, N=6000):
    ph=(np.arange(N)+.5)*2*np.pi/N; d=2*np.pi/N
    x=np.stack([R*np.cos(ph),R*np.sin(ph)],1); t=np.stack([-np.sin(ph),np.cos(ph)],1)
    tot=0.0
    for k in range(1,N):
        dd=np.linalg.norm(x-np.roll(x,-k,0),axis=1); cs=np.sum(t*np.roll(t,-k,0),1)
        dd=np.sqrt(dd**2+r0**2)          # regularised core
        tot+=np.sum(cs/dd)
    return (1/(8*np.pi))*tot*(R*d)**2     # mu=b=1, screw-like (nu=0) Neumann form, half-sum convention
r0=1e-2
for R in (0.5,1.0,2.0):
    E=E_neumann(R,r0); an=(R/2)*(np.log(8*R/r0)-2)/2   # same normalisation /2
    print(f'R={R}: E_neumann={E:.5f}  ratio to (R/4)(ln 8R/r0 - C), C=2 : {E/an:.4f}')
h=1e-3; R=1.0
E=E_neumann(R,r0); dE=(E_neumann(R+h,r0)-E_neumann(R-h,r0))/(2*h)
Lam=np.log(8*R/r0); Ceff=Lam-1/(R*dE/E-1)
print(f'R dE/dR / E = {R*dE/E:.4f} ; fitted C_eff = {Ceff:.3f}')
bavg=1-(R*dE/E)/3
print(f'<beta> nonlocal = {bavg:.4f}  vs local 2/3 = {2/3:.4f}')
ok('<beta> = 2/3 - 1/(3(Lam-C))', bavg-(2/3-1/(3*(Lam-Ceff))), 1e-6)
for lam in (5,10,20,40):
    print(f'Lambda={lam}: <beta>={2/3-1/(3*(lam-Ceff)):.4f}, chiral w/eps needed={1+1/(lam-Ceff):.4f}')